# MedNorm-VI — VietMed-NER Parquet Preprocessing (Colab CPU, data prep only)

**Not model training.** CPU runtime is sufficient (GPU unnecessary). Converts the
VietMed-NER Parquet into canonical half-open JSONL using the versioned mapping;
excludes audio; emits aggregate manifest + hashes only (no raw examples).


## 1. Colab & runtime detection · 2. GPU/RAM/disk report


In [ ]:
import sys
import platform
import shutil

IN_COLAB = 'google.colab' in sys.modules
print('in_colab', IN_COLAB, 'python', platform.python_version())
total, used, free = shutil.disk_usage('/')
print('disk_free_gb', round(free / 1e9, 1))
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print('gpu', torch.cuda.get_device_name(0), 'vram_gb', round(p.total_memory / 1e9, 1))
except ImportError as exc:
    print('torch not installed yet:', exc)


## 3. Configurable Google Drive roots (edit PROJECT_ROOT only)


In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/mednorm-vi'   # <-- EDIT THIS
DATA_ROOT = PROJECT_ROOT + '/data'
MODEL_CACHE = PROJECT_ROOT + '/model_cache'
CHECKPOINT_ROOT = PROJECT_ROOT + '/checkpoints/full_v1'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError as exc:
    print('not in Colab:', exc)


## 4. Pinned dependencies


In [ ]:
PINS = ["pyarrow==17.0.0"]
# !pip -q install ' '.join(PINS)
print('pins', PINS)


## 5. VietMed source path (configurable Drive path)


In [ ]:
import os

VIETMED_PARQUET_DIR = DATA_ROOT + '/external/public_ner/vietmed_ner/data'
ARTIFACT_DIR = DATA_ROOT + '/derived/training_corpora/vietmed_ner_v1'
MAPPING = 'configs/resources/label_mappings/public_ner/vietmed_ner_v1.yaml'
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print('reads', VIETMED_PARQUET_DIR)


## 6. Read + validate rows (schema, words/tags alignment, label inventory)


In [ ]:
import pyarrow.parquet as pq

SPLITS = {'train': 'train-00000-of-00001.parquet',
          'validation': 'validation-00000-of-00001.parquet',
          'test': 'test-00000-of-00001.parquet'}
for split, fname in SPLITS.items():
    t = pq.read_table(VIETMED_PARQUET_DIR + '/' + fname)
    cols = t.column_names
    assert 'words' in cols and 'tags' in cols, cols
    words = t.column('words').to_pylist()
    tags = t.column('tags').to_pylist()
    mism = sum(1 for w, g in zip(words, tags, strict=False) if len(w) != len(g))
    print(split, 'rows', t.num_rows, 'len_mismatch', mism)  # audio column intentionally ignored


## 7. Apply mapping · convert to canonical half-open offsets · exclude audio


In [ ]:
# Reuse the repo adapter contract: BIO words/tags -> joined-token text, half-open
# offsets, DRUGCHEMICAL -> MEDICATION (approximate). Emit canonical JSONL rows with
# {text, entities:[{text,start,end,target_type,source_label,mapping_status,...}]}.
# Assert text[start:end] == entity.text for every emitted entity. Do NOT copy audio.
print('convert VietMed rows -> canonical JSONL (see corpus_build canonical schema)')


## 8. Emit aggregate manifest + hashes (no raw examples)


In [ ]:
# Write ARTIFACT_DIR/canonical_examples/vietmed_ner_examples.jsonl and a manifest
# with row counts, label inventory hash, offset-invalid count (must be 0), and the
# JSONL sha256. Keep only aggregate metadata in any tracked/returned log.
print('write ARTIFACT_DIR manifest + jsonl sha256')


## 9. Return-to-repository instructions
Copy `ARTIFACT_DIR` into `data/derived/training_corpora/vietmed_ner_v1/` in the repo
(git-ignored). Then rebuild the governed corpus to include VietMed:
`python -m mednorm_vi.data_engine.cli build-governed-corpus` (once the VietMed adapter
path is wired) and re-run leakage/offset validation. No audio is copied back.
